# ARC-AGI-2 2026 — P0 Production Notebook

This notebook keeps the original Qwen3-4B + per-puzzle LoRA + augmented DFS pipeline, while applying submission-safety fixes only.

## P0 changes

- All deep public-evaluation diagnostic cells were removed from the production path.
- Every execution gets a clean, run-specific output and status directory.
- Attempt 1 comes from Probmul; attempt 2 is the best distinct Kgmon candidate, with deterministic fallback ordering.
- Missing outputs receive precise status reasons such as `NO_COMPLETED_EOS`, `INVALID_PARSE_ONLY`, `TIMEOUT_NO_COMPLETED_EOS`, or `PUZZLE_EXCEPTION`.
- Submission generation is safe for missing, empty, invalid, duplicate, or corrupt results.
- File loading, puzzle ordering, candidate grouping, and score ties are deterministic.
- GPU count and tokenizer IDs are asserted before expensive inference.
- A compact post-run audit is retained. It is ground-truth-free in competition reruns.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import time

RUNTIME_ROOT = Path("/kaggle/working/arc_p0_runtime")
if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + f"_pid{os.getpid()}"
RUN_DIR = RUNTIME_ROOT / RUN_ID
OUTPUT_DIR = RUN_DIR / "outputs"
STATUS_DIR = RUN_DIR / "status"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
STATUS_DIR.mkdir(parents=True, exist_ok=False)

os.environ["ARC_RUN_ID"] = RUN_ID
os.environ["ARC_OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["ARC_STATUS_DIR"] = str(STATUS_DIR)
os.environ.setdefault("ARC_NPROCS", "4")

global_end_time = 0.0  # starter.py computes a fresh 12-hour deadline

manifest = {
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "output_dir": str(OUTPUT_DIR),
    "status_dir": str(STATUS_DIR),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "nprocs": int(os.environ["ARC_NPROCS"]),
}
with open(RUN_DIR / "manifest.json", "w") as handle:
    json.dump(manifest, handle, indent=2, sort_keys=True)
print(json.dumps(manifest, indent=2))


{
  "run_id": "20260805T065937Z_pid149",
  "run_dir": "/kaggle/working/arc_p0_runtime/20260805T065937Z_pid149",
  "output_dir": "/kaggle/working/arc_p0_runtime/20260805T065937Z_pid149/outputs",
  "status_dir": "/kaggle/working/arc_p0_runtime/20260805T065937Z_pid149/status",
  "created_utc": "2026-08-05T06:59:37.165483+00:00",
  "nprocs": 4
}


In [2]:
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0


In [3]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    if not (isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)):
        return False
    if np.any((guess < 0) | (guess > 9)):
        return False
    return True
    
def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    if sorted(permutation) != list(range(10)):
        raise ValueError(f"Invalid permutation descriptor: {descriptor}")
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            # Reject empty or jagged grids
            if not by_rows:
                return None
            row_len = len(by_rows[0])
            if not all(len(r) == row_len for r in by_rows):
                return None
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except Exception as e:
            print(f"[DEBUG] convert_tokens_to_array failed: {type(e).__name__}: {e}")
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies=None, keys=None, is_orig=False):
        if replies is None:
            replies = {}
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))
    @staticmethod
    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not query['train']:
                    break
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    @staticmethod
    def _fallback_pair(test_input):
        """Return two valid, deterministic and preferably distinct fallback grids."""
        try:
            identity = np.asarray(test_input, dtype=int)
            if not is_valid_solution(identity):
                raise ValueError("invalid test input shape")
        except Exception:
            identity = np.zeros((1, 1), dtype=int)

        alternate = np.zeros_like(identity, dtype=int)
        if np.array_equal(identity, alternate):
            alternate = np.ones_like(identity, dtype=int)
        return identity.tolist(), alternate.tolist()

    @staticmethod
    def _normalize_guesses(guesses):
        valid = []
        seen = set()
        if guesses is None:
            return valid
        for guess in guesses:
            try:
                arr = np.asarray(guess, dtype=int)
            except Exception:
                continue
            if not is_valid_solution(arr):
                continue
            key = tuple(map(tuple, arr.tolist()))
            if key in seen:
                continue
            seen.add(key)
            valid.append(arr.tolist())
        return valid

    def get_submission(self, results=None):
        assert self.is_orig is True, 'Must be run on original dataset.'
        submission = {}
        for key in self.keys:
            submission[key] = []
            for test_case in self.queries[key]['test']:
                attempt_1, attempt_2 = self._fallback_pair(test_case['input'])
                submission[key].append({
                    'attempt_1': attempt_1,
                    'attempt_2': attempt_2,
                })
        if results is not None:
            self.fill_submission(results, submission)
        return submission

    @classmethod
    def fill_submission(cls, results, submission):
        print(f'*** Generating submission for {len(results)} decoded outputs...')
        for key in sorted(results):
            guesses = results[key]
            try:
                base_id, base_nr = key.rsplit('_', 1)
                test_index = int(base_nr)
                target_dict = submission[base_id][test_index]
            except (ValueError, KeyError, IndexError) as exc:
                print(f"[Submission] skip unknown result key {key!r}: {exc}")
                continue

            valid = cls._normalize_guesses(guesses)
            if not valid:
                print(f"[Submission] {key}: no valid decoded guess; keeping safe fallbacks")
                continue

            target_dict['attempt_1'] = valid[0]
            if len(valid) >= 2:
                target_dict['attempt_2'] = valid[1]
            elif target_dict['attempt_2'] == valid[0]:
                # Preserve two distinct attempts whenever possible.
                arr = np.asarray(valid[0], dtype=int)
                alt = np.zeros_like(arr, dtype=int)
                if np.array_equal(arr, alt):
                    alt = np.ones_like(arr, dtype=int)
                target_dict['attempt_2'] = alt.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

Writing arc_loader.py


In [4]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np


def hashable(guess):
    arr = np.asarray(guess, dtype=int)
    return tuple(map(tuple, arr.tolist()))


def _grid_tiebreak(grid_key):
    rows = len(grid_key)
    cols = len(grid_key[0]) if rows else 0
    return rows, cols, grid_key


def score_sum(guesses, getter):
    """Group identical solutions and rank deterministically."""
    grouped = {}
    for guess_id in sorted(guesses):
        guess = guesses[guess_id]
        h = hashable(guess["solution"])
        if h not in grouped:
            grouped[h] = [[], np.asarray(guess["solution"], dtype=int)]
        grouped[h][0].append(guess)

    ranked = []
    for h in sorted(grouped, key=_grid_tiebreak):
        samples, solution = grouped[h]
        score = float(getter(samples))
        if not np.isfinite(score):
            score = float("-inf")
        ranked.append((score, h, solution))

    ranked.sort(key=lambda item: (-item[0], _grid_tiebreak(item[1])))
    return [item[2] for item in ranked]


def getter_full_probmul_3(guesses, baseline=None):
    if baseline is None:
        beam_scores = [g["beam_score"] for g in guesses]
        if not beam_scores:
            return 0.0
        baseline = max(beam_scores) + 1.0
    inf_score = np.sum([baseline - g["beam_score"] for g in guesses])
    aug_score = np.mean([
        np.sum([baseline - s for s in g.get("score_aug", [])])
        for g in guesses
    ])
    return float(inf_score + aug_score)


def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)


def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_values = [
        float(np.mean(g.get("score_aug", [999.0])))
        for g in guesses
    ]
    aug_score = float(np.mean(aug_values)) if aug_values else 999.0
    return float(inf_score - aug_score)


def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [score_full_probmul_3, score_kgmon]


class ArcDecoder:

    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}
        self.load_stats = {
            "files_seen": 0,
            "files_loaded": 0,
            "files_skipped": 0,
            "files_failed": 0,
            "samples_loaded": 0,
            "samples_skipped": 0,
        }
        self.selection_diagnostics = {}

    def load_decoded_results(self, store, run_name=""):
        if not os.path.isdir(store):
            print(f"[Decoder] output directory not found: {store}")
            return self.load_stats

        for filename in sorted(os.listdir(store)):
            path = os.path.join(store, filename)
            if not os.path.isfile(path):
                self.load_stats["files_skipped"] += 1
                continue
            self.load_stats["files_seen"] += 1
            try:
                with bz2.BZ2File(path) as handle:
                    outputs = pickle.load(handle)
                if not isinstance(outputs, list):
                    raise TypeError(f"expected list, got {type(outputs).__name__}")
            except Exception as exc:
                self.load_stats["files_failed"] += 1
                print(f"[Decoder] failed to load {filename}: {type(exc).__name__}: {exc}")
                continue

            base_key = filename.split(".")[0]
            bucket = self.decoded_results.setdefault(base_key, {})
            for index, sample in enumerate(outputs):
                try:
                    if not isinstance(sample, dict):
                        raise TypeError("sample is not a dict")
                    solution = np.asarray(sample["solution"], dtype=int)
                    if solution.ndim != 2 or not all(0 < size <= 30 for size in solution.shape):
                        raise ValueError(f"invalid solution shape {solution.shape}")
                    if np.any((solution < 0) | (solution > 9)):
                        raise ValueError("solution contains values outside 0..9")
                    beam_score = float(sample["beam_score"])
                    score_aug = [float(value) for value in sample.get("score_aug", [])]
                    if not np.isfinite(beam_score) or not all(np.isfinite(score_aug)):
                        raise ValueError("non-finite score")
                    normalized = dict(sample)
                    normalized["solution"] = solution
                    normalized["beam_score"] = beam_score
                    normalized["score_aug"] = score_aug or [999.0] * 8
                except Exception as exc:
                    self.load_stats["samples_skipped"] += 1
                    print(
                        f"[Decoder] skip malformed sample {filename}#{index}: "
                        f"{type(exc).__name__}: {exc}"
                    )
                    continue
                sample_id = f"{filename}{run_name}.out{index}"
                bucket[sample_id] = normalized
                self.load_stats["samples_loaded"] += 1
            self.load_stats["files_loaded"] += 1

        print(f"[Decoder] load stats: {self.load_stats}")
        return self.load_stats

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {
            base_key: selection_algorithm(dict(guesses))
            for base_key, guesses in sorted(self.decoded_results.items())
        }

    def run_dual_selection(self):
        """Use Probmul for attempt 1 and Kgmon for a genuinely cross-ranker attempt 2.

        If Kgmon has no candidate distinct from attempt 1, fall back to the next
        unique Probmul candidate. Empty candidate sets are returned as [], and the
        submission layer supplies deterministic safe fallbacks.
        """
        results = {}
        diagnostics = {}

        for base_key, guesses in sorted(self.decoded_results.items()):
            ranked_probmul = score_full_probmul_3(dict(guesses))
            ranked_kgmon = score_kgmon(dict(guesses))

            attempt_1 = ranked_probmul[0] if ranked_probmul else (
                ranked_kgmon[0] if ranked_kgmon else None
            )
            attempt_1_key = hashable(attempt_1) if attempt_1 is not None else None

            attempt_2 = None
            attempt_2_source = None
            for candidate in ranked_kgmon:
                if hashable(candidate) != attempt_1_key:
                    attempt_2 = candidate
                    attempt_2_source = "kgmon"
                    break
            if attempt_2 is None:
                for candidate in ranked_probmul:
                    if hashable(candidate) != attempt_1_key:
                        attempt_2 = candidate
                        attempt_2_source = "probmul_fallback"
                        break

            chosen = []
            if attempt_1 is not None:
                chosen.append(attempt_1)
            if attempt_2 is not None:
                chosen.append(attempt_2)
            results[base_key] = chosen

            diagnostics[base_key] = {
                "raw_samples": len(guesses),
                "probmul_unique": len(ranked_probmul),
                "kgmon_unique": len(ranked_kgmon),
                "attempt_1_source": "probmul" if ranked_probmul else (
                    "kgmon" if ranked_kgmon else "none"
                ),
                "attempt_2_source": attempt_2_source or "submission_fallback",
                "selected_count": len(chosen),
                "selected_distinct": len({hashable(x) for x in chosen}) == len(chosen),
            }

        self.selection_diagnostics = diagnostics
        return results

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")
        labels = {}
        num_tasks_per_puzzle = {}

        for basekey in sorted(self.decoded_results):
            mult_key, mult_sub = basekey.rsplit("_", 1)
            num_tasks_per_puzzle[mult_key] = max(
                num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1
            )
            if basekey in self.dataset.replies:
                labels[basekey] = self.dataset.replies[basekey][0]

        num_puzzles = len(num_tasks_per_puzzle)
        if not labels:
            print("[Benchmark] no ground-truth replies available; skipped")
            return

        for selection_algorithm in selection_algorithms:
            selected = self.run_selection_algo(selection_algorithm)
            correct = {
                key for key, guesses in selected.items()
                if key in labels and any(
                    np.array_equal(guess, labels[key])
                    for guess in guesses[:self.n_guesses]
                )
            }
            score = sum(
                1 / num_tasks_per_puzzle[key.rsplit("_", 1)[0]]
                for key in correct
            )
            print(f"acc: {score:5.2f}/{num_puzzles:3} ({selection_algorithm.__name__})")

        selected = self.run_dual_selection()
        correct = {
            key for key, guesses in selected.items()
            if key in labels and any(
                np.array_equal(guess, labels[key])
                for guess in guesses[:self.n_guesses]
            )
        }
        score = sum(
            1 / num_tasks_per_puzzle[key.rsplit("_", 1)[0]]
            for key in correct
        )
        print(f"acc: {score:5.2f}/{num_puzzles:3} (cross-ranker dual selection)")


Writing arc_decoder.py


In [5]:
%%writefile arc_solver.py

import os
os.environ["UNSLOTH_DISABLE_RL_PATCH"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import copy
import queue
import json
import torch
import numpy as np
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

import hashlib

def deterministic_seed(s, mod=10000):
    """
    Python's built-in hash() is randomized by PYTHONHASHSEED,
    so it gives different values across processes/runs.
    This function provides a consistent, reproducible seed across all runs.
    """
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % mod

logging.disable(logging.WARNING)

# ------------------------------------------------------------
# Fixed token IDs (as used in fine-tuning)
# ------------------------------------------------------------
ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

MAX_BEAM_CANDIDATES = 10
class UnslothFixedTrainer(UnslothTrainer):
    # Issue https://github.com/unslothai/unsloth/issues/2435
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            # Ensure labels is 1D (avoid scalar tensor)
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)
            # Use torch.nonzero instead of torch.where to avoid bool scalar issues
            user_mask = (labels == USER_TOKEN_ID)
            assistant_mask = (labels == ASSISTANT_TOKEN_ID)
            user_start_idx = torch.nonzero(user_mask, as_tuple=True)[0].tolist()
            assistant_start_idx = torch.nonzero(assistant_mask, as_tuple=True)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            eos_mask = (labels == EOS_ID)
            end_idx = torch.nonzero(eos_mask, as_tuple=True)[0].tolist()
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                if start >= end:
                    continue
                if j % 2 == 1:  # assistant response part
                    start += 2
                    end += 1
                    if start < end and end <= labels.size(0):
                        batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time):
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i].sort(key=lambda x: x[0])
        candidates[i] = candidates[i][:MAX_BEAM_CANDIDATES]

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000.0)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        # Direct cache pass (fast, no clone)
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_cache = outputs.past_key_values

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=next_cache,
            start_time=start_time,
            end_time=end_time,
        )

        del outputs
        # torch.cuda.empty_cache() removed for speed

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x: x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    
    max_len = max(batch_lengths)
    padded_tokens = []
    attention_masks = []
    for tokens in batch_tokens:
        pad_len = max_len - len(tokens)
        padded = tokens + [PAD_ID] * pad_len
        padded_tokens.append(padded)
        attention_masks.append([1] * len(tokens) + [0] * pad_len)
        
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    attention_mask = torch.tensor(attention_masks, device=model.device, dtype=torch.long)
    
    # FIX: use_cache=False because padding breaks KV cache
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True, use_cache=False)
    
    result = []
    # FIX: Slicing on GPU before transferring to CPU to prevent massive OOM Crash
    for i, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_logits = outputs.logits[i, query_length-1 : query_length-1+len(answer_tokens), :]
        answer_log_probs = answer_logits.float().log_softmax(-1).cpu()
        answer_score = answer_log_probs[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
        
    return result


def _env_flag(name):
    return os.getenv(name, "").strip().lower() in {"1", "true", "yes", "y", "on"}


def assert_tokenizer_contract(tokenizer):
    """Fail fast if the local model tokenizer no longer matches hard-coded IDs."""
    expected_single_tokens = {
        "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
        "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
        "\n": 10,
        "<|im_start|>user\n": USER_TOKEN_ID,
        "<|im_start|>assistant\n": ASSISTANT_TOKEN_ID,
        "<|im_end|>": EOS_ID,
    }
    errors = []
    for text, expected_id in expected_single_tokens.items():
        actual = tokenizer.encode(text, add_special_tokens=False)
        if actual != [expected_id]:
            errors.append(f"{text!r}: expected [{expected_id}], got {actual}")
    if tokenizer.pad_token_id != PAD_ID:
        errors.append(f"pad_token_id: expected {PAD_ID}, got {tokenizer.pad_token_id}")
    if errors:
        raise RuntimeError("Tokenizer contract mismatch: " + "; ".join(errors))
    print("[Tokenizer] contract verified", flush=True)


def worker(rank, p_queue, end_time):
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)

    if not torch.cuda.is_available():
        raise RuntimeError(f"Rank {rank}: CUDA is not available")
    if torch.cuda.device_count() != 1:
        raise RuntimeError(
            f"Rank {rank}: expected exactly one visible GPU after CUDA_VISIBLE_DEVICES, "
            f"found {torch.cuda.device_count()}"
        )

    rerun_mode = _env_flag("KAGGLE_IS_COMPETITION_RERUN")
    output_dir = os.environ.get("ARC_OUTPUT_DIR", "/kaggle/working/arc_outputs")
    status_dir = os.environ.get("ARC_STATUS_DIR", "/kaggle/working/arc_status")
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(status_dir, exist_ok=True)

    def write_status(base_key, payload):
        payload = dict(payload)
        payload.update({
            "base_key": base_key,
            "rank": rank,
            "written_at": time.time(),
        })
        final_path = os.path.join(status_dir, f"{base_key}.json")
        tmp_path = final_path + f".rank{rank}.tmp"
        with open(tmp_path, "w") as handle:
            json.dump(payload, handle, sort_keys=True)
        os.replace(tmp_path, final_path)

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch_fused",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )
    assert_tokenizer_contract(tokenizer)

    model = FastLanguageModel.get_peft_model(model, **peft_params)
    for _, parameter in model.named_parameters():
        if parameter.dtype == torch.float32:
            parameter.data = parameter.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {key: value.clone().detach() for key, value in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)
    formatter = QwenFormatter(tokenizer=tokenizer)
    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(0.2)

    test_path = (
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        if rerun_mode else
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
    )
    arc_test_set = ArcDataset.from_file(test_path)

    def make_fallback_results(base_key, reason):
        try:
            puzzle_id, test_id = base_key.rsplit("_", 1)
            test_input = np.asarray(
                arc_test_set.queries[puzzle_id]["test"][int(test_id)]["input"],
                dtype=int,
            )
            identity = test_input.copy()
        except Exception:
            identity = np.zeros((1, 1), dtype=int)
        alternate = np.zeros_like(identity, dtype=int)
        if np.array_equal(identity, alternate):
            alternate = np.ones_like(identity, dtype=int)
        return [
            {
                "beam_score": 999.0,
                "score_aug": [999.0] * 8,
                "solution": identity,
                "is_fallback": True,
                "fallback_reason": reason,
            },
            {
                "beam_score": 1000.0,
                "score_aug": [1000.0] * 8,
                "solution": alternate,
                "is_fallback": True,
                "fallback_reason": reason,
            },
        ]

    def save_fallback_result(base_key, reason):
        fallback_path = os.path.join(output_dir, f"{base_key}.fallback")
        with bz2.BZ2File(fallback_path, "w") as handle:
            pickle.dump(make_fallback_results(base_key, reason), handle)
        print(f"[Rank {rank}] FALLBACK_SAVE {base_key}: {reason}", flush=True)

    while True:
        if time.time() > end_time:
            print(f"[Rank {rank}] global deadline reached", flush=True)
            break

        try:
            key = p_queue.get(timeout=60)
        except queue.Empty:
            continue
        if key is None:
            break

        start_time = time.time()
        base_keys = [
            f"{key}_{index}"
            for index in range(len(arc_test_set.queries[key]["test"]))
        ]
        diagnostics = {
            base_key: {
                "puzzle_id": key,
                "expected_augments": 0,
                "scheduled_augments": 0,
                "dfs_returned_augments": 0,
                "completed_beams": 0,
                "invalid_parse_beams": 0,
                "decode_exceptions": 0,
                "score_exceptions": 0,
                "valid_candidates": 0,
                "saved_candidate_files": 0,
                "dfs_exceptions": 0,
                "no_eos_local_timeout_augments": 0,
                "no_eos_global_deadline_augments": 0,
                "no_eos_pruned_or_token_limit_augments": 0,
                "timed_out": False,
                "errors": [],
            }
            for base_key in base_keys
        }

        try:
            torch.cuda.reset_peak_memory_stats()
            set_peft_model_state_dict(model, default_weights.copy(), adapter_name="default")
            model = FastLanguageModel.for_training(model)

            puzzle_ds = arc_test_set.change_keys([key])
            train_seed = deterministic_seed(key)
            print(f"[Rank {rank}] train_seed={train_seed} puzzle={key}", flush=True)
            train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=train_seed)
            train_ds = train_ds.cut_to_len(
                formatter=formatter,
                name="text",
                max_len=max_seq_length,
            )

            with io.StringIO() as buffer, redirect_stdout(buffer), redirect_stderr(buffer):
                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )
                stats = trainer.train()
                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
                del trainer

            model = FastLanguageModel.for_inference(model)
            gc.collect()
            torch.cuda.empty_cache()
            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] training peak={memory_allocated}MB puzzle={key}", flush=True)
            print(f"[Rank {rank}] training stats puzzle={key}: {stats}", flush=True)
            torch.cuda.reset_peak_memory_stats()

            puzzle_ds_multi = puzzle_ds.split_multi_replies()
            eval_seed = deterministic_seed(f"{key}_eval")
            eval_ds = puzzle_ds_multi.augment(n=2, seed=eval_seed)
            eval_ds = eval_ds.cut_to_len(
                formatter=formatter,
                name="input",
                max_len=max_seq_length - max_new_tokens,
            )

            test_id_to_subkeys = defaultdict(list)
            for subkey in sorted(eval_ds.keys):
                base_key = subkey.split(".")[0]
                diagnostics[base_key]["expected_augments"] += 1
                test_id = base_key.rsplit("_", 1)[1]
                test_id_to_subkeys[test_id].append(subkey)

            batches = []
            for test_id in sorted(test_id_to_subkeys, key=int):
                subkeys = test_id_to_subkeys[test_id]
                for offsets in ([0, 4], [2, 6], [8, 12], [10, 14]):
                    batch = []
                    for offset in offsets:
                        batch.extend(subkeys[offset:offset + 2])
                    if batch:
                        batches.append(batch)

            with torch.inference_mode():
                known_scores = {}
                for subkeys in batches:
                    spend_time = time.time() - start_time
                    if spend_time > 1600 or time.time() > end_time:
                        for base_key in base_keys:
                            diagnostics[base_key]["timed_out"] = True
                        print(f"[Rank {rank}] timeout after {spend_time:.1f}s puzzle={key}", flush=True)
                        break

                    print(f"[Rank {rank}] decoding {subkeys}", flush=True)
                    token_groups = defaultdict(list)
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        input_tokens = tokenizer.encode(data["input"])
                        token_groups[len(input_tokens)].append((subkey, input_tokens))
                        diagnostics[subkey.split(".")[0]]["scheduled_augments"] += 1

                    for token_len, group_items in sorted(token_groups.items()):
                        group_subkeys = [item[0] for item in group_items]
                        group_tokens = [item[1] for item in group_items]
                        try:
                            dfs_result, dfs_metadata = inference_turbo_dfs(
                                model, group_tokens, max_new_tokens, max_score, end_time
                            )
                        except Exception as exc:
                            message = f"DFS_EXCEPTION {type(exc).__name__}: {exc}"
                            print(
                                f"[Rank {rank}] {message} token_len={token_len} "
                                f"subkeys={group_subkeys}",
                                flush=True,
                            )
                            for subkey in group_subkeys:
                                base_key = subkey.split(".")[0]
                                diagnostics[base_key]["dfs_exceptions"] += 1
                                diagnostics[base_key]["errors"].append(message[:500])
                            continue

                        returned_ids = {subkey_id for subkey_id, _ in dfs_result}
                        for group_id, subkey in enumerate(group_subkeys):
                            if group_id in returned_ids:
                                continue
                            diag = diagnostics[subkey.split(".")[0]]
                            if dfs_metadata["hit_global_deadline"]:
                                diag["no_eos_global_deadline_augments"] += 1
                            elif dfs_metadata["hit_local_time_budget"]:
                                diag["no_eos_local_timeout_augments"] += 1
                            else:
                                diag["no_eos_pruned_or_token_limit_augments"] += 1

                        for subkey_id, scored_beams in dfs_result:
                            subkey = group_subkeys[subkey_id]
                            base_key = subkey.split(".")[0]
                            diag = diagnostics[base_key]
                            diag["dfs_returned_augments"] += 1
                            diag["completed_beams"] += len(scored_beams)
                            decoded_result = []

                            for beam_score, beam_tokens in scored_beams:
                                try:
                                    array = formatter.convert_tokens_to_array(beam_tokens)
                                    if array is None:
                                        diag["invalid_parse_beams"] += 1
                                        continue
                                    solution = puzzle_ds_multi.invert_mod(
                                        array, subkey, inv_perm=True
                                    )
                                except Exception as exc:
                                    diag["decode_exceptions"] += 1
                                    diag["errors"].append(
                                        f"DECODE_EXCEPTION {type(exc).__name__}: {exc}"[:500]
                                    )
                                    continue

                                grid_id = (base_key, tuple(map(tuple, solution)))
                                if grid_id in known_scores:
                                    augmented_scores = known_scores[grid_id]
                                else:
                                    try:
                                        aug_dataset = ArcDataset(
                                            keys=[base_key],
                                            queries={base_key: puzzle_ds_multi.queries.get(base_key)},
                                            replies={base_key: [solution.tolist()]},
                                        )
                                        aug_seed = deterministic_seed(
                                            f"{base_key}_aug", mod=1024**2
                                        )
                                        aug_dataset = aug_dataset.augment(seed=aug_seed)
                                        aug_dataset = aug_dataset.cut_to_len(
                                            formatter=formatter,
                                            name="input",
                                            max_len=max_seq_length - max_new_tokens,
                                        )
                                        aug_queries = []
                                        aug_answers = []
                                        for sample in aug_dataset.as_list(formatter):
                                            aug_queries.append(sample["input"])
                                            aug_answers.append(sample["reply"])
                                        augmented_scores = (
                                            calc_scores(
                                                aug_queries[:4], aug_answers[:4], tokenizer, model
                                            )
                                            + calc_scores(
                                                aug_queries[4:], aug_answers[4:], tokenizer, model
                                            )
                                        )
                                    except Exception as exc:
                                        diag["score_exceptions"] += 1
                                        diag["errors"].append(
                                            f"SCORE_EXCEPTION {type(exc).__name__}: {exc}"[:500]
                                        )
                                        augmented_scores = [999.0] * 8
                                    known_scores[grid_id] = augmented_scores

                                decoded_result.append({
                                    "beam_score": beam_score,
                                    "score_aug": augmented_scores,
                                    "solution": solution,
                                    "is_fallback": False,
                                })
                                diag["valid_candidates"] += 1

                            if decoded_result:
                                with bz2.BZ2File(
                                    os.path.join(output_dir, subkey), "w"
                                ) as handle:
                                    pickle.dump(decoded_result, handle)
                                diag["saved_candidate_files"] += 1

            for base_key in base_keys:
                diag = diagnostics[base_key]
                if diag["saved_candidate_files"] > 0:
                    reason = "SUCCESS"
                elif diag["no_eos_global_deadline_augments"] > 0 and diag["dfs_returned_augments"] == 0:
                    reason = "GLOBAL_DEADLINE_NO_COMPLETED_EOS"
                elif diag["timed_out"] and diag["dfs_returned_augments"] == 0:
                    reason = "PUZZLE_TIME_BUDGET_NO_COMPLETED_EOS"
                elif diag["no_eos_local_timeout_augments"] > 0 and diag["dfs_returned_augments"] == 0:
                    reason = "DFS_LOCAL_TIMEOUT_NO_COMPLETED_EOS"
                elif diag["dfs_exceptions"] > 0 and diag["dfs_returned_augments"] == 0:
                    reason = "DFS_EXCEPTION_NO_COMPLETION"
                elif diag["scheduled_augments"] == 0:
                    reason = "NO_AUGMENT_SCHEDULED"
                elif diag["dfs_returned_augments"] == 0 and diag["no_eos_pruned_or_token_limit_augments"] > 0:
                    reason = "ALL_BRANCHES_PRUNED_OR_TOKEN_LIMIT_NO_EOS"
                elif diag["dfs_returned_augments"] == 0:
                    reason = "NO_COMPLETED_EOS_UNCLASSIFIED"
                elif diag["valid_candidates"] == 0 and diag["invalid_parse_beams"] > 0:
                    reason = "INVALID_PARSE_ONLY"
                elif diag["valid_candidates"] == 0 and diag["decode_exceptions"] > 0:
                    reason = "DECODE_EXCEPTION_ONLY"
                else:
                    reason = "EMPTY_CANDIDATE_POOL"

                diag["final_reason"] = reason
                diag["elapsed_seconds"] = round(time.time() - start_time, 3)
                if reason != "SUCCESS":
                    save_fallback_result(base_key, reason)
                write_status(base_key, diag)

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            spend_time = time.time() - start_time
            print(
                f"[Rank {rank}] finished {key} in {spend_time:.1f}s "
                f"inference_peak={memory_allocated}MB",
                flush=True,
            )

        except Exception as exc:
            message = f"PUZZLE_EXCEPTION {type(exc).__name__}: {exc}"
            print(f"[Rank {rank}] {message} puzzle={key}", flush=True)
            for base_key in base_keys:
                diag = diagnostics[base_key]
                diag["errors"].append(message[:500])
                if diag["saved_candidate_files"] > 0:
                    diag["final_reason"] = "PARTIAL_PUZZLE_EXCEPTION_WITH_CANDIDATES"
                else:
                    diag["final_reason"] = "PUZZLE_EXCEPTION"
                diag["elapsed_seconds"] = round(time.time() - start_time, 3)
                try:
                    if diag["saved_candidate_files"] == 0:
                        save_fallback_result(base_key, "PUZZLE_EXCEPTION")
                    write_status(base_key, diag)
                except Exception as fallback_exc:
                    print(
                        f"[Rank {rank}] FALLBACK_FAILED {base_key}: "
                        f"{type(fallback_exc).__name__}: {fallback_exc}",
                        flush=True,
                    )
            gc.collect()
            torch.cuda.empty_cache()
            continue


Writing arc_solver.py


In [6]:
%%writefile starter.py
import os
import time
import json
import argparse
import traceback
import numpy as np
import torch
import torch.multiprocessing as mp

RUN_FULL_EVAL = True
DEBUG_PUZZLES = []
NPROCS = int(os.getenv("ARC_NPROCS", "4"))


def env_flag(name):
    return os.getenv(name, "").strip().lower() in {"1", "true", "yes", "y", "on"}


def get_puzzle_priority(key, data):
    train = data[key]["train"]
    n_train = len(train)
    avg_input_size = np.mean([
        np.prod(np.shape(example["input"])) for example in train
    ])
    size_ratios = [
        max(np.prod(np.shape(example["output"])), np.prod(np.shape(example["input"])))
        / max(min(np.prod(np.shape(example["output"])), np.prod(np.shape(example["input"]))), 1)
        for example in train
    ]
    max_ratio = max(size_ratios)
    avg_ratio = np.mean(size_ratios)
    n_colors = len({
        int(color)
        for example in train
        for grid in [example["input"], example["output"]]
        for color in np.unique(grid).tolist()
    })
    is_dynamic = any(
        np.shape(example["input"]) != np.shape(example["output"])
        for example in train
    )

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30
        + (avg_ratio - 1) / 10 * 0.25
        + n_colors / 10 * 0.20
        + (0.25 if is_dynamic else 0.0)
    )
    if n_train >= 6 and tier < 3:
        fine *= 0.7
    return tier, float(fine)


def local_worker(rank, work_queue, end_time, import_lock):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["UNSLOTH_DISABLE_RL_PATCH"] = "1"
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
    try:
        if torch.cuda.device_count() != 1:
            raise RuntimeError(
                f"Rank {rank}: expected one visible GPU after CUDA_VISIBLE_DEVICES; "
                f"found {torch.cuda.device_count()}"
            )
        with import_lock:
            print(f"[Rank {rank}] importing arc_solver...", flush=True)
            from arc_solver import worker
            print(f"[Rank {rank}] arc_solver import ok", flush=True)
        worker(rank, work_queue, end_time)
    except Exception as exc:
        print(f"[Rank {rank}] WORKER_EXCEPTION {type(exc).__name__}: {exc}", flush=True)
        traceback.print_exc()
        raise


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()
    end_time = args.end_time if args.end_time > 0 else time.time() + 12 * 3600

    available_gpus = torch.cuda.device_count()
    if NPROCS <= 0:
        raise RuntimeError(f"ARC_NPROCS must be positive, got {NPROCS}")
    if available_gpus < NPROCS:
        raise RuntimeError(
            f"Requested {NPROCS} workers but only {available_gpus} CUDA GPUs are available"
        )
    print(f"[Starter] GPU assertion passed: {available_gpus} available, {NPROCS} requested", flush=True)

    rerun_mode = env_flag("KAGGLE_IS_COMPETITION_RERUN")
    test_path = (
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        if rerun_mode else
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
    )
    with open(test_path, "r") as handle:
        data = json.load(handle)

    keys_sorted = sorted(
        data,
        key=lambda key: (*get_puzzle_priority(key, data), key),
    )
    selected_keys = [
        key for key in keys_sorted
        if rerun_mode or RUN_FULL_EVAL or key in DEBUG_PUZZLES
    ]
    if not selected_keys:
        raise RuntimeError("No puzzles selected")

    tiers = [get_puzzle_priority(key, data)[0] for key in selected_keys]
    print(
        f"[Starter] puzzles={len(selected_keys)} easy={tiers.count(1)} "
        f"hard={tiers.count(2)} extreme={tiers.count(3)} "
        f"rerun_mode={rerun_mode}",
        flush=True,
    )
    print(f"[Starter] first={selected_keys[0]} last={selected_keys[-1]}", flush=True)

    manager = mp.Manager()
    work_queue = manager.Queue()
    import_lock = manager.Lock()
    for key in selected_keys:
        work_queue.put(key)
    for _ in range(NPROCS):
        work_queue.put(None)

    mp.spawn(
        local_worker,
        args=(work_queue, end_time, import_lock),
        nprocs=NPROCS,
        join=True,
    )


Writing starter.py


In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env.update({
    "UNSLOTH_DISABLE_RL_PATCH": "1",
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
    "OMP_NUM_THREADS": "12",
})
subprocess.run(
    [sys.executable, "starter.py", "--end-time", str(global_end_time)],
    check=True,
    env=env,
)


[Starter] GPU assertion passed: 4 available, 4 requested
[Starter] puzzles=120 easy=65 hard=51 extreme=4 rerun_mode=False
[Starter] first=36a08778 last=4c7dc4dd
[Rank 1] importing arc_solver...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 1] arc_solver import ok
[Rank 3] importing arc_solver...
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:  50%|█████     | 1/2 [00:37<00:37, 37.53s/it]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 3] arc_solver import ok
[Rank 2] importing arc_solver...
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.40s/it]


[Rank 3] WORKER_EXCEPTION RuntimeError: Tokenizer contract mismatch: '<|im_start|>user\n': expected [11], got [14, 11, 10]; '<|im_start|>assistant\n': expected [12], got [14, 12, 10]
[Rank 1] WORKER_EXCEPTION RuntimeError: Tokenizer contract mismatch: '<|im_start|>user\n': expected [11], got [14, 11, 10]; '<|im_start|>assistant\n': expected [12], got [14, 12, 10]


Traceback (most recent call last):
  File "/kaggle/working/starter.py", line 77, in local_worker
    worker(rank, work_queue, end_time)
  File "/kaggle/working/arc_solver.py", line 355, in worker
    assert_tokenizer_contract(tokenizer)
  File "/kaggle/working/arc_solver.py", line 271, in assert_tokenizer_contract
    raise RuntimeError("Tokenizer contract mismatch: " + "; ".join(errors))
RuntimeError: Tokenizer contract mismatch: '<|im_start|>user\n': expected [11], got [14, 11, 10]; '<|im_start|>assistant\n': expected [12], got [14, 12, 10]
Traceback (most recent call last):
  File "/kaggle/working/starter.py", line 77, in local_worker
    worker(rank, work_queue, end_time)
  File "/kaggle/working/arc_solver.py", line 355, in worker
    assert_tokenizer_contract(tokenizer)
  File "/kaggle/working/arc_solver.py", line 271, in assert_tokenizer_contract
    raise RuntimeError("Tokenizer contract mismatch: " + "; ".join(errors))
RuntimeError: Tokenizer contract mismatch: '<|im_start|>use

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path
import numpy as np

from arc_loader import ArcDataset, is_valid_solution
from arc_decoder import ArcDecoder


def env_flag(name):
    return os.getenv(name, "").strip().lower() in {"1", "true", "yes", "y", "on"}

rerun_mode = env_flag("KAGGLE_IS_COMPETITION_RERUN")
challenge_path = (
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    if rerun_mode else
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
)

data = ArcDataset.from_file(challenge_path)
decoder_dataset = data.split_multi_replies()
decoder = ArcDecoder(decoder_dataset, n_guesses=2)
decoder.load_decoded_results(os.environ["ARC_OUTPUT_DIR"])
selected = decoder.run_dual_selection()
submission = data.get_submission(selected)

# Competition-required file location.
with open("submission.json", "w") as handle:
    json.dump(submission, handle, sort_keys=True)

# Compact, rerun-safe production audit. No hidden labels are used.
expected_subkeys = {
    f"{task_id}_{test_index}"
    for task_id in data.keys
    for test_index in range(len(data.queries[task_id]["test"]))
}
loaded_subkeys = set(decoder.decoded_results)
status_records = []
for path in sorted(Path(os.environ["ARC_STATUS_DIR"]).glob("*.json")):
    try:
        with open(path) as handle:
            status_records.append(json.load(handle))
    except Exception as exc:
        print(f"[Audit] failed to read {path.name}: {type(exc).__name__}: {exc}")

reason_counts = Counter(
    record.get("final_reason", "UNKNOWN") for record in status_records
)
duplicate_attempts = 0
invalid_attempts = 0
for task_id, task_outputs in submission.items():
    for output in task_outputs:
        attempt_1 = np.asarray(output["attempt_1"], dtype=int)
        attempt_2 = np.asarray(output["attempt_2"], dtype=int)
        duplicate_attempts += int(np.array_equal(attempt_1, attempt_2))
        invalid_attempts += int(not is_valid_solution(attempt_1))
        invalid_attempts += int(not is_valid_solution(attempt_2))

selection_source_counts = Counter(
    row["attempt_2_source"] for row in decoder.selection_diagnostics.values()
)
audit = {
    "run_id": os.environ.get("ARC_RUN_ID"),
    "rerun_mode": rerun_mode,
    "expected_subkeys": len(expected_subkeys),
    "loaded_subkeys": len(loaded_subkeys),
    "missing_loaded_subkeys": sorted(expected_subkeys - loaded_subkeys),
    "unexpected_loaded_subkeys": sorted(loaded_subkeys - expected_subkeys),
    "status_files": len(status_records),
    "missing_status_subkeys": sorted(
        expected_subkeys - {record.get("base_key") for record in status_records}
    ),
    "final_reason_counts": dict(sorted(reason_counts.items())),
    "attempt_2_source_counts": dict(sorted(selection_source_counts.items())),
    "duplicate_submission_attempts": duplicate_attempts,
    "invalid_submission_attempts": invalid_attempts,
    "decoder_load_stats": decoder.load_stats,
    "submission_tasks": len(submission),
}
with open(Path(os.environ["ARC_STATUS_DIR"]).parent / "production_audit.json", "w") as handle:
    json.dump(audit, handle, indent=2, sort_keys=True)

# Reload validation catches JSON serialization or schema corruption.
with open("submission.json") as handle:
    reloaded_submission = json.load(handle)
assert set(reloaded_submission) == set(data.keys)
assert audit["invalid_submission_attempts"] == 0
print(json.dumps(audit, indent=2, sort_keys=True))
print("[Production] submission.json written successfully")

# The only label-aware diagnostic retained, and it is impossible to run on private reruns.
if not rerun_mode:
    data.load_replies(
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"
    )
    decoder.dataset = data.split_multi_replies()
    decoder.benchmark_selection_algos()
    print("*** Reload score:", data.validate_submission(reloaded_submission))
